# Week 04 - Homework: EU AI Act RAG Evaluation

You will build a comprehensive RAG evaluation system for EU AI Act compliance questions using the concepts from Week 04 Lesson 01 - 03.

### What You'll Build

- **EU AI Act RAG System**: Document retrieval and answer generation for compliance questions
- **Evaluation Framework**: Multi-dimensional assessment using 4 key evaluators
- **Target Function**: Wrapper function for systematic evaluation
- **Comprehensive Testing**: Full evaluation pipeline with metrics

### Background: EU AI Act

The EU AI Act is a comprehensive regulatory framework for artificial intelligence in the European Union. It establishes rules for AI systems based on their risk levels and includes requirements for transparency, accountability, and human oversight.

>**TODO**: Complete the evaluation sections based on Lesson 03

---


## Part 1: Environment Setup and Dependencies


In [2]:
# Install required dependencies
%pip install -U --quiet langsmith langchain langchain-google-genai langchain-community python-dotenv tiktoken pypdf requests langgraph streamlit pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import necessary libraries for the RAG evaluation system
import os
import requests
from typing_extensions import Annotated, TypedDict
from dotenv import load_dotenv

# Document processing and vector store
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Gemini API integration (replacing OpenAI)
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Agent and tool creation
from langchain.tools import Tool

# LangSmith for tracing and evaluation
from langsmith import Client, traceable
from langsmith.evaluation import evaluate
from langsmith.schemas import Run, Example

# LangGraph for agent creation
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

print("✓ All libraries imported successfully!")

ModuleNotFoundError: No module named 'google.protobuf'

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Set up LangSmith environment variables for tracing and evaluation
os.environ['LANGSMITH_TRACING'] = os.getenv('LANGSMITH_TRACING', 'true')
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = os.getenv('LANGSMITH_PROJECT', 'AI-Act-RAG-System')
os.environ['LANGSMITH_ENDPOINT'] = os.getenv('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')

# Get Gemini API key from environment
gemini_api_key = os.getenv('GEMINI_API_KEY')

# Verify that necessary API keys are loaded
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found in .env file. Please add it to continue.")

if not os.getenv('LANGSMITH_API_KEY'):
    raise ValueError("LANGSMITH_API_KEY not found in .env file. Please add it to continue.")

# Initialize LangSmith client for dataset management and evaluation
client = Client()

print("✓ Environment configured successfully!")
print(f"✓ LangSmith Project: {os.getenv('LANGSMITH_PROJECT')}")
print(f"✓ Using Gemini API for language model operations")

## Part 2: EU AI Act Document Processing

First, we'll download and process the EU AI Act document to create our knowledge base.


In [ ]:
# Load EU AI Act PDF document
def load_eu_ai_act_pdf():
    """Load the EU AI Act PDF document."""
    pdf_path = "eu_ai_act.pdf"
    
    if not os.path.exists(pdf_path):
        print(f"Error: PDF file not found at {pdf_path}")
        print("Please ensure the eu_ai_act.pdf file is in the current directory")
        return None
    
    # Load PDF using PyPDFLoader
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    print(f"✓ Loaded EU AI Act PDF with {len(documents)} pages")
    return documents

# Load the document
documents = load_eu_ai_act_pdf()
if documents:
    print("✓ EU AI Act PDF loaded successfully!")
else:
    print("✗ Failed to load EU AI Act PDF")


In [ ]:
# Process the EU AI Act document and create a searchable knowledge base
def create_knowledge_base(documents):
    """
    Transforms the EU AI Act PDF into a searchable knowledge base.
    
    This function:
    1. Splits the documents into manageable chunks
    2. Creates embeddings for each chunk using Gemini
    3. Stores the embeddings in a vector database for quick retrieval
    
    Args:
        documents: List of loaded PDF documents
        
    Returns:
        vectorstore: An in-memory vector store containing document embeddings
    """
    
    if not documents:
        print("Error: No documents provided")
        return None
    
    # Split documents into chunks for better retrieval
    # Using tiktoken encoder ensures consistent chunk sizes
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=1000,      # Maximum characters per chunk
        chunk_overlap=200      # Overlap between chunks to preserve context
    )
    
    doc_splits = text_splitter.split_documents(documents)
    print(f"✓ Created {len(doc_splits)} document chunks")
    
    # Create vector store using Gemini embeddings
    # Embeddings convert text into numerical representations for similarity search
    vectorstore = InMemoryVectorStore.from_documents(
        documents=doc_splits,
        embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    )
    
    return vectorstore

# Create knowledge base from loaded documents
if documents:
    vectorstore = create_knowledge_base(documents)
    # Create a retriever that will fetch the top 4 most relevant chunks for each query
    retriever = vectorstore.as_retriever(k=4)
    print("✓ Knowledge base created successfully!")
else:
    print("✗ Cannot create knowledge base without documents")

## Part 3: RAG Implementation

Now we'll create an agent that can reason about and answer questions regarding the EU AI Act.


In [ ]:
# Initialize the Gemini language model for our RAG system
# Using a low temperature (0.1) ensures more consistent, factual responses
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",  # Fast and efficient model for production use
    temperature=0.1,            # Low temperature for factual, consistent responses
    convert_system_message_to_human=True  # Gemini compatibility setting
)

# Create RAG tool that searches the EU AI Act document
@traceable  # Enables LangSmith tracing for debugging and evaluation
def rag_search(query: str) -> str:
    """
    Searches the EU AI Act document for relevant information.
    
    This function:
    1. Takes a user question
    2. Retrieves the most relevant document chunks
    3. Returns the combined context for the agent to use
    
    Args:
        query: The search question
        
    Returns:
        A formatted string containing relevant EU AI Act information
    """
    try:
        # Retrieve the top 4 most relevant document chunks
        docs = retriever.invoke(query)
        
        # Combine the chunks into a single context string
        context = "\n\n".join([doc.page_content for doc in docs])
        
        return f"Relevant information from EU AI Act:\n{context}"
    except Exception as e:
        return f"Error searching documents: {str(e)}"

# Wrap the search function as a LangChain Tool
# Tools allow agents to take specific actions during their reasoning process
rag_tool = Tool(
    name="eu_ai_act_search",
    description="Search the EU AI Act document for information about AI regulations, compliance requirements, and legal provisions. Use this when you need specific information from the EU AI Act.",
    func=rag_search
)

print("✓ RAG tool created successfully!")

In [ ]:
# Create a ReAct (Reasoning and Acting) agent
# ReAct agents alternate between reasoning about the problem and taking actions

# Use LangGraph's prebuilt ReAct agent with memory checkpointing
# Memory allows the agent to maintain conversation context across multiple turns
agent_executor = create_react_agent(
    model=llm,                          # The Gemini language model
    tools=[rag_tool],                   # Available tools (our EU AI Act search)
    checkpointer=MemorySaver()          # Saves conversation state
)

print("✓ ReAct agent created successfully!")
print("✓ The agent can now reason about EU AI Act questions and search the document when needed")

In [ ]:
# Test the agent with a sample EU AI Act question
@traceable  # Track this execution in LangSmith
def test_agent(question: str) -> str:
    """
    Tests the ReAct agent with a compliance question.
    
    The agent will:
    1. Understand the question
    2. Decide if it needs to search the document
    3. Use the RAG tool to find relevant information
    4. Reason about the retrieved information
    5. Formulate a comprehensive answer
    
    Args:
        question: The EU AI Act compliance question
        
    Returns:
        The agent's answer based on the document
    """
    try:
        # Create a unique thread ID for this conversation
        config = {"configurable": {"thread_id": "test_thread"}}
        
        # Invoke the agent with the question
        response = agent_executor.invoke({"messages": question}, config=config)
        
        # Extract the final answer from the response
        return response["messages"][-1].content
    except Exception as e:
        return f"Error: {str(e)}"

# Test with a sample question about high-risk AI systems
test_question = "What are the requirements for obligations of deployers of high-risk AI systems under the EU AI Act?"
print(f"Question: {test_question}")
print("\n" + "="*80)
response = test_agent(test_question)
print(f"Answer: {response}")

## Part 4: RAG Evaluation Framework

Now we'll implement comprehensive RAG evaluation using the techniques from Week 04 Lesson 02.


In [ ]:
# Create evaluation dataset for EU AI Act RAG
# This dataset contains questions with expected answers for comprehensive evaluation

evaluation_examples = [
    {
        "inputs": {"question": "What are the prohibited AI practices under the EU AI Act?"},
        "outputs": {
            "expected_answer": "The EU AI Act prohibits AI practices that pose unacceptable risks, including: (1) AI systems that deploy subliminal techniques to materially distort behavior causing harm, (2) AI systems exploiting vulnerabilities of specific groups causing harm, (3) biometric categorization systems using sensitive characteristics, (4) social scoring systems by public authorities, (5) AI systems assessing risk of committing criminal offenses based solely on profiling, (6) creating or expanding facial recognition databases through untargeted scraping, (7) emotion recognition in workplace and educational institutions (with limited exceptions), (8) real-time remote biometric identification in publicly accessible spaces for law enforcement (with limited exceptions)."
        },
        "metadata": {"category": "prohibited_practices"}
    },
    {
        "inputs": {"question": "What is a high-risk AI system according to the EU AI Act?"},
        "outputs": {
            "expected_answer": "A high-risk AI system is one that falls into specific categories listed in Annex III of the EU AI Act or is used as a safety component of products covered by Union harmonization legislation. High-risk AI systems include those used in: biometric identification, critical infrastructure management, education and vocational training, employment and worker management, access to essential services, law enforcement, migration and border control management, and administration of justice. These systems pose significant risks to health, safety, or fundamental rights."
        },
        "metadata": {"category": "definitions"}
    },
    {
        "inputs": {"question": "What are the requirements for providers of high-risk AI systems?"},
        "outputs": {
            "expected_answer": "Providers of high-risk AI systems must: (1) establish and maintain a risk management system, (2) ensure appropriate data governance and quality, (3) prepare technical documentation, (4) implement automatic logging capabilities, (5) provide clear and adequate information to users, (6) design systems for human oversight, (7) ensure appropriate levels of accuracy, robustness, and cybersecurity, (8) establish a quality management system, (9) conduct conformity assessments, (10) register the system in the EU database, (11) take corrective actions when needed, and (12) cooperate with authorities during investigations."
        },
        "metadata": {"category": "compliance"}
    },
    {
        "inputs": {"question": "What is the conformity assessment procedure for high-risk AI systems?"},
        "outputs": {
            "expected_answer": "The conformity assessment for high-risk AI systems involves either: (1) Internal control procedure - where the provider verifies compliance based on technical documentation and quality management system, or (2) Assessment by a notified body - for certain systems listed in Annex III. The procedure includes checking compliance with requirements in Chapter 2, preparing technical documentation, implementing a quality management system, and maintaining records. After successful assessment, the provider affixes the CE marking and draws up an EU declaration of conformity."
        },
        "metadata": {"category": "procedures"}
    },
    {
        "inputs": {"question": "What are the transparency requirements for AI systems?"},
        "outputs": {
            "expected_answer": "The EU AI Act establishes transparency obligations including: (1) AI systems intended to interact with people must be designed to inform users they are interacting with AI (unless obvious from context), (2) emotion recognition and biometric categorization systems must inform users when they are being used, (3) AI-generated content (deep fakes) must be disclosed and marked as artificially generated or manipulated, (4) users of AI systems that generate or manipulate text, audio, or video content must disclose that the content is AI-generated, and (5) providers must ensure outputs are marked in a machine-readable format when technically feasible."
        },
        "metadata": {"category": "transparency"}
    },
    {
        "inputs": {"question": "What are the obligations of deployers of high-risk AI systems?"},
        "outputs": {
            "expected_answer": "Deployers of high-risk AI systems must: (1) use the system according to instructions of use, (2) ensure human oversight, (3) monitor the system's operation and inform the provider or distributor of incidents and malfunctions, (4) keep logs automatically generated by the system for at least 6 months (or longer as specified), (5) conduct a data protection impact assessment when required by GDPR, (6) ensure input data is relevant and representative, (7) cooperate with authorities, and (8) for certain systems like those used in employment or law enforcement, inform affected persons about the use of the AI system."
        },
        "metadata": {"category": "compliance"}
    }
]

print(f"✓ Created {len(evaluation_examples)} evaluation examples with reference answers")
print("✓ Each example includes question, expected answer, and category metadata")

In [ ]:
# Create dataset in LangSmith
dataset_name = "eu-ai-act-rag-evaluation"

try:
    # Create dataset
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="EU AI Act RAG evaluation dataset with compliance questions and reference answers"
    )
    
    # Add examples to dataset
    client.create_examples(
        dataset_id=dataset.id,
        examples=evaluation_examples
    )
    
    print(f"✓ Dataset '{dataset_name}' created successfully!")
    print(f"Dataset ID: {dataset.id}")
    
except Exception as e:
    if "already exists" in str(e):
        # Use existing dataset
        dataset = client.read_dataset(dataset_name=dataset_name)
        print(f"✓ Using existing dataset: {dataset.id}")
    else:
        print(f"Error creating dataset: {e}")


---

## Part 5: Individual Evaluators Implementation

### TODO: Copy the 4 evaluators from Lesson 03
1. **Correctness Evaluator**: Copy the schema, instructions, and function from Lesson 03
2. **Relevance Evaluator**: Copy the schema, instructions, and function from Lesson 03  
3. **Groundedness Evaluator**: Copy the schema, instructions, and function from Lesson 03
4. **Retrieval Relevance Evaluator**: Copy the schema, instructions, and function from Lesson 03

These four evaluation techniques assess different dimensions of RAG system performance:
- **Correctness**: Response vs reference answer
- **Relevance**: Response vs input question  
- **Groundedness**: Response vs retrieved documents
- **Retrieval Relevance**: Retrieved documents vs input question

In [ ]:
# CORRECTNESS EVALUATOR
# Measures how well the generated answer matches the expected reference answer
# This helps us verify that our RAG system provides accurate responses

# Define the structure for correctness grading
class CorrectnessGrade(TypedDict):
    """Schema for evaluating answer correctness."""
    score: Annotated[int, ..., "Correctness score: 0 (incorrect) or 1 (correct)"]
    reasoning: Annotated[str, ..., "Explanation for the correctness score"]

# Instructions for the grading model
correctness_instructions = """You are evaluating the correctness of an answer to an EU AI Act compliance question.

Compare the generated answer with the reference answer:
- Score 1 (Correct): The generated answer captures the same key information and meaning as the reference answer, even if worded differently
- Score 0 (Incorrect): The generated answer is wrong, incomplete, or significantly differs from the reference answer

Provide:
1. A score (0 or 1)
2. Clear reasoning explaining your score

Focus on semantic correctness, not exact wording."""

# Initialize a separate Gemini model for grading
# Using a more powerful model for evaluation ensures accurate assessments
grader_llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro",  # More capable model for evaluation
    temperature=0,            # Zero temperature for consistent grading
    convert_system_message_to_human=True
).with_structured_output(CorrectnessGrade)

def correctness(run: Run, example: Example) -> dict:
    """
    Evaluates if the generated answer is correct compared to the reference answer.
    
    This evaluator:
    1. Takes the generated answer from the RAG system
    2. Compares it to the expected reference answer
    3. Returns a score (0 or 1) and reasoning
    
    Args:
        run: The execution run containing the generated answer
        example: The test example containing the reference answer
        
    Returns:
        A dictionary with the score and reasoning
    """
    # Extract the generated answer from the run outputs
    generated_answer = run.outputs.get("answer", "")
    
    # Extract the reference answer from the example
    reference_answer = example.outputs.get("expected_answer", "")
    
    # Grade the answer using the LLM
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": f"Reference Answer: {reference_answer}\n\nGenerated Answer: {generated_answer}"}
    ])
    
    return {
        "key": "correctness",
        "score": grade["score"],
        "reason": grade["reasoning"]
    }

print("✓ Correctness Evaluator implemented successfully!")

In [ ]:
# RELEVANCE EVALUATOR
# Measures how well the generated answer addresses the input question
# Even a correct answer can be irrelevant if it doesn't address what was asked

# Define the structure for relevance grading
class RelevanceGrade(TypedDict):
    """Schema for evaluating answer relevance to the question."""
    score: Annotated[int, ..., "Relevance score: 0 (not relevant) or 1 (relevant)"]
    reasoning: Annotated[str, ..., "Explanation for the relevance score"]

# Instructions for relevance grading
relevance_instructions = """You are evaluating whether an answer is relevant to an EU AI Act compliance question.

Assess if the generated answer directly addresses the question asked:
- Score 1 (Relevant): The answer directly addresses the question and provides pertinent information
- Score 0 (Not Relevant): The answer is off-topic, doesn't address the question, or provides unrelated information

Provide:
1. A score (0 or 1)
2. Clear reasoning explaining your score

Focus on whether the answer addresses what was actually asked."""

# Initialize Gemini model for relevance grading
relevance_llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro",
    temperature=0,
    convert_system_message_to_human=True
).with_structured_output(RelevanceGrade)

def relevance(run: Run, example: Example) -> dict:
    """
    Evaluates if the generated answer is relevant to the input question.
    
    This evaluator:
    1. Takes the user's question
    2. Takes the generated answer
    3. Determines if the answer actually addresses the question
    
    Args:
        run: The execution run containing the generated answer
        example: The test example containing the input question
        
    Returns:
        A dictionary with the score and reasoning
    """
    # Extract the input question
    question = example.inputs.get("question", "")
    
    # Extract the generated answer
    generated_answer = run.outputs.get("answer", "")
    
    # Grade the relevance using the LLM
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": f"Question: {question}\n\nGenerated Answer: {generated_answer}"}
    ])
    
    return {
        "key": "relevance",
        "score": grade["score"],
        "reason": grade["reasoning"]
    }

print("✓ Relevance Evaluator implemented successfully!")

In [ ]:
# GROUNDEDNESS EVALUATOR
# Checks if the answer is supported by the retrieved documents
# Prevents the model from making up information (hallucination)

# Define the structure for groundedness grading
class GroundedGrade(TypedDict):
    """Schema for evaluating answer groundedness in retrieved documents."""
    score: Annotated[int, ..., "Groundedness score: 0 (not grounded) or 1 (grounded)"]
    reasoning: Annotated[str, ..., "Explanation for the groundedness score"]

# Instructions for groundedness grading
grounded_instructions = """You are evaluating whether an answer is grounded in (supported by) the retrieved documents from the EU AI Act.

Check if all claims in the answer are supported by the retrieved documents:
- Score 1 (Grounded): All statements in the answer are supported by or can be inferred from the retrieved documents
- Score 0 (Not Grounded): The answer contains information not found in the retrieved documents, or makes unsupported claims

Provide:
1. A score (0 or 1)
2. Clear reasoning explaining your score

Be strict: the answer should only contain information from the retrieved documents."""

# Initialize Gemini model for groundedness grading
grounded_llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro",
    temperature=0,
    convert_system_message_to_human=True
).with_structured_output(GroundedGrade)

def groundedness(run: Run, example: Example) -> dict:
    """
    Evaluates if the generated answer is grounded in the retrieved documents.
    
    This evaluator:
    1. Takes the documents that were retrieved from the EU AI Act
    2. Takes the generated answer
    3. Checks if every claim in the answer is supported by the documents
    
    This prevents hallucination and ensures factual accuracy.
    
    Args:
        run: The execution run containing the answer and retrieved documents
        example: The test example (not used here but required by interface)
        
    Returns:
        A dictionary with the score and reasoning
    """
    # Extract the generated answer
    generated_answer = run.outputs.get("answer", "")
    
    # Extract the retrieved documents
    documents = run.outputs.get("documents", [])
    
    # Combine documents into a single context string
    context = "\n\n".join([doc if isinstance(doc, str) else doc.get("content", "") for doc in documents])
    
    # Grade the groundedness using the LLM
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": f"Retrieved Documents:\n{context}\n\nGenerated Answer: {generated_answer}"}
    ])
    
    return {
        "key": "groundedness",
        "score": grade["score"],
        "reason": grade["reasoning"]
    }

print("✓ Groundedness Evaluator implemented successfully!")

In [ ]:
# RETRIEVAL RELEVANCE EVALUATOR
# Checks if the retrieved documents are relevant to the input question
# Good retrieval is the foundation of a good RAG system

# Define the structure for retrieval relevance grading
class RetrievalRelevanceGrade(TypedDict):
    """Schema for evaluating relevance of retrieved documents to the question."""
    score: Annotated[int, ..., "Retrieval relevance score: 0 (not relevant) or 1 (relevant)"]
    reasoning: Annotated[str, ..., "Explanation for the retrieval relevance score"]

# Instructions for retrieval relevance grading
retrieval_relevance_instructions = """You are evaluating whether retrieved documents from the EU AI Act are relevant to a compliance question.

Assess if the retrieved documents contain information that could help answer the question:
- Score 1 (Relevant): The documents contain information pertinent to answering the question
- Score 0 (Not Relevant): The documents don't contain useful information for answering the question

Provide:
1. A score (0 or 1)
2. Clear reasoning explaining your score

Focus on whether the documents contain information that could help answer the question, even if they don't contain the complete answer."""

# Initialize Gemini model for retrieval relevance grading
retrieval_relevance_llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro",
    temperature=0,
    convert_system_message_to_human=True
).with_structured_output(RetrievalRelevanceGrade)

def retrieval_relevance(run: Run, example: Example) -> dict:
    """
    Evaluates if the retrieved documents are relevant to the input question.
    
    This evaluator:
    1. Takes the user's question
    2. Takes the documents that were retrieved
    3. Checks if the documents contain information useful for answering the question
    
    This helps diagnose retrieval problems in the RAG pipeline.
    
    Args:
        run: The execution run containing the retrieved documents
        example: The test example containing the input question
        
    Returns:
        A dictionary with the score and reasoning
    """
    # Extract the input question
    question = example.inputs.get("question", "")
    
    # Extract the retrieved documents
    documents = run.outputs.get("documents", [])
    
    # Combine documents into a single context string
    context = "\n\n".join([doc if isinstance(doc, str) else doc.get("content", "") for doc in documents])
    
    # Grade the retrieval relevance using the LLM
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": f"Question: {question}\n\nRetrieved Documents:\n{context}"}
    ])
    
    return {
        "key": "retrieval_relevance",
        "score": grade["score"],
        "reason": grade["reasoning"]
    }

print("✓ Retrieval Relevance Evaluator implemented successfully!")

---

## Part 6: Testing and Evaluation Execution

### TODO: Test evaluators and run comprehensive evaluation
1. **Individual Testing**: Test each evaluator with sample questions
2. **Full Evaluation**: Run comprehensive evaluation using all evaluators
3. **Results Analysis**: Interpret and analyze the evaluation results


In [ ]:
# TARGET FUNCTION FOR RAG EVALUATION
# This function wraps our RAG system for systematic evaluation
# It takes a question, runs it through the RAG system, and returns both the answer and retrieved documents

@traceable  # Enable LangSmith tracing for this function
def target_function(inputs: dict) -> dict:
    """
    Wraps the EU AI Act RAG system for evaluation.
    
    This function:
    1. Takes an input question from the evaluation dataset
    2. Retrieves relevant documents from the EU AI Act
    3. Generates an answer using the ReAct agent
    4. Returns both the answer and documents for evaluation
    
    The evaluators will use this output to assess:
    - Correctness: Is the answer correct?
    - Relevance: Does the answer address the question?
    - Groundedness: Is the answer supported by the documents?
    - Retrieval Relevance: Are the documents relevant to the question?
    
    Args:
        inputs: Dictionary containing the 'question' key
        
    Returns:
        Dictionary with 'answer' and 'documents' keys
    """
    # Extract the question from inputs
    question = inputs.get("question", "")
    
    # Retrieve relevant documents from the EU AI Act
    # This is what the RAG tool does internally, so we do it here to capture the documents
    docs = retriever.invoke(question)
    
    # Convert documents to a list of dictionaries for evaluation
    documents = [{"content": doc.page_content, "metadata": doc.metadata} for doc in docs]
    
    # Generate answer using the ReAct agent
    config = {"configurable": {"thread_id": f"eval_{hash(question)}"}}
    response = agent_executor.invoke({"messages": question}, config=config)
    
    # Extract the final answer from the agent's response
    answer = response["messages"][-1].content
    
    # Return both answer and documents for comprehensive evaluation
    return {
        "answer": answer,
        "documents": documents
    }

print("✓ Target function created successfully!")
print("✓ This function can now be used to evaluate the RAG system systematically")

In [ ]:
# TEST INDIVIDUAL EVALUATORS
# Before running full evaluation, let's test each evaluator with a sample question
# This helps ensure they work correctly before running the comprehensive evaluation

print("Testing individual evaluators with a sample question...")
print("="*80)

# Test question
test_question = "What are the prohibited AI practices under the EU AI Act?"

# Get the answer and documents using our target function
test_result = target_function({"question": test_question})

print(f"Question: {test_question}")
print(f"\nGenerated Answer: {test_result['answer'][:200]}...")
print(f"\nNumber of retrieved documents: {len(test_result['documents'])}")
print("="*80)

# Create a mock Run and Example for testing evaluators
from langsmith.schemas import Run, Example

# Create mock run with outputs
mock_run = Run(
    id="test-run",
    name="test",
    run_type="chain",
    inputs={"question": test_question},
    outputs=test_result,
    start_time="2025-01-01T00:00:00Z",
    end_time="2025-01-01T00:00:01Z"
)

# Create mock example with reference answer
mock_example = Example(
    id="test-example",
    created_at="2025-01-01T00:00:00Z",
    inputs={"question": test_question},
    outputs={"expected_answer": evaluation_examples[0]["outputs"]["expected_answer"]}
)

# Test each evaluator
print("\n1. Testing Relevance Evaluator...")
relevance_result = relevance(mock_run, mock_example)
print(f"   Score: {relevance_result['score']}")
print(f"   Reason: {relevance_result['reason']}")

print("\n2. Testing Groundedness Evaluator...")
groundedness_result = groundedness(mock_run, mock_example)
print(f"   Score: {groundedness_result['score']}")
print(f"   Reason: {groundedness_result['reason']}")

print("\n3. Testing Retrieval Relevance Evaluator...")
retrieval_result = retrieval_relevance(mock_run, mock_example)
print(f"   Score: {retrieval_result['score']}")
print(f"   Reason: {retrieval_result['reason']}")

print("\n4. Testing Correctness Evaluator...")
correctness_result = correctness(mock_run, mock_example)
print(f"   Score: {correctness_result['score']}")
print(f"   Reason: {correctness_result['reason']}")

print("\n" + "="*80)
print("✓ All evaluators tested successfully!")

In [ ]:
# RUN COMPREHENSIVE EVALUATION
# This runs our RAG system through all test cases and applies all four evaluators
# Results will be visible in LangSmith UI for detailed analysis

print("Running comprehensive RAG evaluation...")
print("="*80)

# Run evaluation with all four evaluators
# The evaluate() function will:
# 1. Run target_function on each example in the dataset
# 2. Apply each evaluator to the results
# 3. Track everything in LangSmith for detailed analysis

experiment_results = evaluate(
    target_function,                    # Our RAG system wrapper
    data=dataset_name,                   # The evaluation dataset we created
    evaluators=[                         # All four evaluators
        correctness,                     # Checks answer vs reference
        relevance,                       # Checks answer vs question
        groundedness,                    # Checks answer vs documents
        retrieval_relevance             # Checks documents vs question
    ],
    experiment_prefix="eu-ai-act-rag",  # Prefix for this evaluation run
    description="Comprehensive RAG evaluation for EU AI Act compliance system using Gemini API"
)

print("="*80)
print("✓ Evaluation completed successfully!")
print(f"\n📊 Results Summary:")
print(f"   - Experiment: {experiment_results['experiment_name']}")
print(f"   - Total examples evaluated: {len(evaluation_examples)}")
print(f"   - Evaluators used: Correctness, Relevance, Groundedness, Retrieval Relevance")
print(f"\n🔗 View detailed results in LangSmith:")
print(f"   https://smith.langchain.com")
print(f"\n💡 Key Insights:")
print(f"   - Check the LangSmith UI for score breakdowns by evaluator")
print(f"   - Review individual traces to see agent reasoning")
print(f"   - Identify patterns in failed evaluations")
print(f"   - Use insights to improve retrieval and answer generation")

---

## Part 7: Results Analysis and Conclusion

### TODO: Analyze your evaluation results
1. **Review LangSmith Results**: Check the evaluation results in the LangSmith UI
2. **Interpret Scores**: Analyze correctness, relevance, groundedness, and retrieval relevance scores
3. **Identify Issues**: Look for patterns in failed evaluations
4. **Document Insights**: Write down key findings and areas for improvement

### Key Takeaways
- **Multi-dimensional Evaluation**: Use multiple evaluators for comprehensive RAG assessment
- **LLM-as-Judge**: Leverage advanced models for sophisticated evaluation
- **Domain Adaptation**: Tailor evaluation criteria to your specific domain (EU AI Act)
- **Continuous Improvement**: Use evaluation insights to enhance your RAG system

**Remember**: RAG evaluation is an ongoing process that requires continuous refinement based on real-world performance and evolving requirements!
